# QuantOne v2 — Phase 0 GPU probe

For each manifest model: download its SMALLEST study quant, verify sha256, load with llama-cpp-python 0.3.33, apply embedded chat template, assert thinking stays off, run 10 sample tasks, measure tok/s. Writes `phase0_gpu_report.json`.

Small tier → Kaggle T4. Big tier (qwen36_27b, gemma4_26b_a4b) → Colab L4 (set `TIER = "big"` there). MoE memory for gemma4_26b_a4b gets measured here, not assumed.

In [ ]:
TIER = "small"   # "small" (Kaggle T4) or "big" (Colab L4/A100)
MODELS = None    # None = all in tier; or a list to re-probe, e.g. ["qwen35_2b","qwen35_4b"]
REPO_GIT_URL = "https://github.com/kaushiksai29/QuantOne.git"
BRANCH = "v2"

In [ ]:
import os, shutil, subprocess

def gpu_name():
    if shutil.which("nvidia-smi") is None:
        return None
    r = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                       capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 and r.stdout.strip() else None

if not os.path.exists("QuantOne"):
    !git clone -b {BRANCH} {REPO_GIT_URL}
%cd QuantOne

GPU = gpu_name()
if GPU is None:
    raise SystemExit(
        "STOP: no GPU in this session. This is the GPU probe — its tok/s feed "
        "the Phase 4 GPU-hour budget, so CPU numbers would mislead it. Turn on "
        "the accelerator (Kaggle: right panel -> Accelerator -> GPU T4; Colab: "
        "Runtime -> Change runtime type -> T4/L4), then Run All. If it was "
        "already on, do Run -> Factory reset to clear the bad install first.")
print("GPU detected:", GPU)

# GPU present -> CUDA wheel; source-build fallback if the prebuilt won't load
!pip install -q llama-cpp-python==0.3.33 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 PyYAML
try:
    import llama_cpp
except Exception as exc:
    print(f"prebuilt CUDA wheel unusable ({exc}); building from source (~15-25 min) ...")
    !pip uninstall -y -q llama-cpp-python
    os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on"
    !pip install -q llama-cpp-python==0.3.33 --no-binary llama-cpp-python
    import llama_cpp
print("llama-cpp-python", llama_cpp.__version__, "| GPU:", GPU)

In [ ]:
import json, hashlib, re, time, urllib.request, os
from pathlib import Path
from llama_cpp import Llama

OUTDIR = "/kaggle/working" if os.path.exists("/kaggle") else "/content/drive/MyDrive" if os.path.exists("/content/drive/MyDrive") else "."
manifest = json.load(open("manifest.json"))
models = {k: m for k, m in manifest["models"].items()
          if m["tier"] == TIER and (MODELS is None or k in MODELS)}
tasks = [json.loads(l) for l in open("tasks/tasks.jsonl")][:10]
THINK_RE = re.compile(r"<think|<thinking|</think", re.I)

def sha256_file(p, chunk=1<<20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def gen(llm, messages, max_tokens=256):
    out = llm.create_chat_completion(messages=messages, temperature=0,
                                     max_tokens=max_tokens, seed=0)
    return out["choices"][0]["message"]["content"], out["usage"]["completion_tokens"]

# Candidate thinking-off mechanisms. `wrap` turns a task prompt into messages.
CANDIDATES = {
    "none": lambda p: [{"role": "user", "content": p}],
    "/no_think system prompt": lambda p: [{"role": "system", "content": "/no_think"},
                                          {"role": "user", "content": p}],
    "'/no_think' appended to user turn": lambda p: [{"role": "user", "content": p + " /no_think"}],
}

def eval_mech(llm, wrap):
    """Run ALL 10 tasks under one mechanism; return (json_shaped, leaks, tok, secs)."""
    t0 = time.perf_counter(); n_tok = parses = leaks = 0
    for t in tasks:
        txt, ct = gen(llm, wrap(t["prompt"]))
        n_tok += ct
        leaks += bool(THINK_RE.search(txt))
        parses += txt.strip().startswith(("{", "`"))
    return parses, leaks, n_tok, time.perf_counter() - t0

report = []
for key, m in models.items():
    quant, info = sorted(m["files"].items(), key=lambda kv: kv[1]["gb"] if isinstance(kv[1], dict) else 9e9)[0]
    url = f"https://huggingface.co/{m['gguf_repo']}/resolve/main/{info['path']}"
    local = Path(info["path"].split("/")[-1])
    print(f"== {key} ({quant}, {info['gb']}GB) ==", flush=True)
    if not local.exists():
        urllib.request.urlretrieve(url, local)
    assert sha256_file(local) == info["sha256"], f"sha mismatch for {key}"
    entry = {"model": key, "quant": quant, "sha256": "OK"}
    try:
        llm = Llama(model_path=str(local), n_ctx=4096, chat_format=None,
                    n_gpu_layers=-1, verbose=False)
        entry["load"] = "OK"

        hybrid = any(w in m.get("thinking_off", "").lower()
                     for w in ("hybrid", "thinking model", "/no_think"))
        # Always evaluate "none" across all 10. If it's not already clean
        # (10/10 json AND 0 leaks) OR the model is a known hybrid, try the
        # /no_think mechanisms too and keep the best. This is what catches a
        # model that looks clean on task 0 but reasons on later tasks.
        results = {}
        results["none"] = eval_mech(llm, CANDIDATES["none"])
        need_more = hybrid or results["none"][0] < 10 or results["none"][1] > 0
        if need_more:
            for name in ("/no_think system prompt", "'/no_think' appended to user turn"):
                results[name] = eval_mech(llm, CANDIDATES[name])
        # best = max json_shaped, then min leaks, prefer "none" on ties
        best = min(results, key=lambda k: (-results[k][0], results[k][1], k != "none"))
        parses, leaks, n_tok, secs = results[best]
        entry.update(
            thinking_off=("none needed" if best == "none" else best),
            json_shaped=f"{parses}/10", think_leaks=f"{leaks}/10",
            tok_per_s=round(n_tok/secs, 1),
            all_mechanisms={k: f"json {v[0]}/10, leaks {v[1]}/10" for k, v in results.items()},
            verdict="OK" if (parses >= 8 and leaks == 0) else "NEEDS REVIEW")
        del llm
    except Exception as e:
        entry["load"] = f"FAIL: {e}"
        entry["verdict"] = "FAIL"
    print("  ", {k: entry[k] for k in entry if k != "all_mechanisms"}, flush=True)
    report.append(entry)
    local.unlink()  # disk budget

out = {"meta": {"tier": TIER, "gpu": GPU, "llama_cpp": llama_cpp.__version__},
       "models": report}
out_path = f"{OUTDIR}/phase0_gpu_report_{TIER}.json"
json.dump(out, open(out_path, "w"), indent=2)
print(f"\nDONE on {GPU} — report at {out_path}")
print("verdicts:", {r['model']: r.get('verdict') for r in report})